In [ ]:
from datetime import datetime
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sorted_portfolios import get_sorted_index, get_sorted_portfolio
from utils import *
from signals import *
from visualization import plot_group_cum, plot_solo_cum

plt.style.use("default")
warnings.filterwarnings("ignore")
%load_ext autoreload
%autoreload 2

# Data Preparation

In [ ]:
# CRSP data
crsp = pd.read_csv("data//csrp_msf.csv", index_col=0)
crsp['date'] = pd.to_datetime(crsp.date)
# Construct dfs
dfs = {}
# Price DataFrame (abs for negative price)
dfs['prc'] = np.abs(date_cusip_pivot(crsp, 'prc_adj'))
# Return DataFrame
dfs['ret'] = get_simple_ret(dfs['prc'])
# Volume Threshold DataFrame. 
# # If the value (i, j) is NaN, the stock(i) at time (j) is excluded in sorting.
dfs['idx_vol_threshold'] = get_volume_threshold(date_cusip_pivot(crsp, "vol"), 0.30)
# Index for June
idx_june = dfs['ret'].index.month == 6
# Index for March, June, September, and December
idx_Q = idx_june.copy()
for month in [3, 9, 12]:
    idx_Q = idx_Q | (dfs['ret'].index.month == month)
# Compustat quarterly data
comp_funq = pd.read_csv("data//comp_fundq.csv", index_col=0).rename(columns={"datadate": "date"})
# Use the first eight codes to match the crsp cusip
comp_funq['cusip'] = comp_funq['cusip'].apply(lambda x: str(x)[:8])
comp_funq['date'] = pd.to_datetime(comp_funq.date)
# Drop the "nan" cusip data
comp_funq = comp_funq[~(comp_funq.cusip == "nan")].reset_index(drop=True)

# Size (Quality Check)

In [ ]:
df_size = comp_funq.copy(deep=True)
df_size['size'] = df_size['mkvaltq']
df_size = get_non_duplicated_df(df_size, 'size')

In [ ]:
# Construct the size for each stock
df_size = get_size_df(comp_funq, crsp)
# We re-balance the portfolio in every June
df_size[~idx_june] = np.nan
df_size[idx_june].head()

In [ ]:
# Get the list of index DataFrame for size
df_size_idx_l = get_sorted_index(df_size * dfs['idx_vol_threshold'])
# The lowest group weights calculated on 1976-06-30
df_size_idx_l[0][idx_june].iloc[1, :].dropna()

In [ ]:
# Get the sorted size portfolio
size_portfolio = get_sorted_portfolio(df_ret=dfs['ret'], df_idx_l=df_size_idx_l)
size_portfolio.tail()

In [ ]:
def process_ff(df):
    df = df / 100
    # Convert index
    df.index = df.index.to_series().apply(lambda x: pd.to_datetime(f"{str(x)[:4]}-{str(x)[4:]}-01") + pd.offsets.MonthEnd(0))
    return df

def ls(df, l, s):
    return df.iloc[:, l] - df.iloc[:, s]


# Fama-French SMB factor
ff_size = process_ff(pd.read_csv('data//Portfolios_Formed_on_ME.CSV', index_col=0))
# Select the sorted decile portfolios
ff_size = ff_size.iloc[:, 9:]

In [ ]:
start = (1980, 7, 1)
end = (2020, 12, 31)
temp = filter_datetime_df(size_portfolio, start, end)
temp_ff = filter_datetime_df(ff_size, start, end)
plt.figure(figsize=(12, 8))
plt.plot(temp.index, (1+ls(temp, 0, -1)).to_numpy().cumprod(), label='Market Value (Low-High)')
plt.plot(temp.index, (1+ls(temp_ff, 0, -1)).to_numpy().cumprod(), label='Fama-French Size (Small-Big)')
corr = np.corrcoef([ls(temp, 0, -1), ls(temp_ff, 0, -1)])[0, 1]
plt.legend(loc='upper right')
plt.title(f"Cumulative Returns: Return Corr = {corr:.2%}")
plt.grid()
plt.xticks(temp.index[::50], rotation=45)
plt.show()

In [ ]:
ff_mom = pd.read_excel("french_mom_ew.xlsx", index_col=0)
ff_mom = process_ff(ff_mom)

In [ ]:
df_mom = get_momentum(dfs['ret'])

In [ ]:
price_momentum = df_mom.copy(deep=True)
price_momentum[~idx_Q] = np.nan
mom_portfolio = get_sorted_portfolio(
    dfs['ret'], get_sorted_index(
        df_mom * get_volume_threshold(date_cusip_pivot(crsp, "vol"), 0.2),
        holding_period=3
        #quantiles=np.array([0.2, 0.4, 0.6, 0.8])
        )
)
plot_group_cum(mom_portfolio, "Price Momentum")
plot_solo_cum(ls(mom_portfolio, -1, 0), "Price Momentum LS (High-Low) Cumulative Return")

In [ ]:
start = (1975, 7, 1)
end = (2020, 12, 31)
temp_mom = filter_datetime_df(mom_portfolio, start, end)
temp_mom_ff = filter_datetime_df(ff_mom, start, end)
plt.figure(figsize=(12, 8))

plt.plot(temp_mom.index, (1+ls(temp_mom, -1, 0)).to_numpy().cumprod(), label='Momentum (High-Low)')
plt.plot(temp_mom.index, (1+ls(temp_mom_ff, -1, 0)).to_numpy().cumprod(), label='Fama-French Momentum (High-Low)')
# plt.plot(temp_mom.index, (1+temp_mom.iloc[:, 0]).to_numpy().cumprod(), label='Small')
# plt.plot(temp_mom.index, (1+temp_mom_ff.iloc[:, 0]).to_numpy().cumprod(), label='FF Small')
# plt.plot(temp_mom.index, (1+temp_mom.iloc[:, -1]).to_numpy().cumprod(), '--', label='Big')
# plt.plot(temp_mom.index, (1+temp_mom_ff.iloc[:,-1]).to_numpy().cumprod(), '--', label='FF Big')
corr = np.corrcoef([ls(temp_mom, 0, -1), ls(temp_mom_ff, 0, -1)])[0, 1]
plt.legend(loc='upper right')
plt.title(f"Cumulative Returns: Return Corr = {corr:.2%}")
plt.xticks(temp.index[::50], rotation=45)
plt.grid()
plt.show()

In [ ]:
size = comp_funq["date cusip mkvaltq consol".split()]
size['size'] = size.eval("mkvaltq")
size = get_non_duplicated_df(size, "size")
size = date_cusip_pivot(size, "size").reindex(columns=dfs['ret'].columns)
size = get_time_merge(size, dfs['ret'])

In [ ]:
size_v1 = size.copy(deep=True)
size_v1[~idx_june] = np.nan
size_port1 = get_sorted_portfolio(
    dfs['ret'], get_sorted_index(
        size_v1 * get_volume_threshold(date_cusip_pivot(crsp, "vol"), 0.2),
        holding_period=12
        #,quantiles=np.array([0.2, 0.4, 0.6, 0.8])
        #,quantiles=np.array([.3, .7])
        )
)
size_port1 = filter_datetime_df(size_port1, (2003, 1, 1), (2023, 12, 31))
plot_group_cum(size_port1, "Size")
plot_solo_cum(ls(size_port1, 0, -1), "SMB")

In [ ]:
start = (2005, 7, 1)
end = (2020, 12, 31)
temp = filter_datetime_df(size_port1, start, end)
temp_ff = filter_datetime_df(ff_size, start, end)
plt.figure(figsize=(12, 8))

plt.plot(temp.index, (1+ls(temp, 0, -1)).to_numpy().cumprod(), label='Market Value (Low-High)')
plt.plot(temp.index, (1+ls(temp_ff, 0, -1)).to_numpy().cumprod(), label='Fama-French Size (Small-Big)')
# plt.plot(temp.index, (1+temp.iloc[:, 0]).to_numpy().cumprod(), label='Small')
# plt.plot(temp.index, (1+temp_ff.iloc[:, 0]).to_numpy().cumprod(), label='FF Small')
# plt.plot(temp.index, (1+temp.iloc[:, -1]).to_numpy().cumprod(), '--', label='Big')
# plt.plot(temp.index, (1+temp_ff.iloc[:,-1]).to_numpy().cumprod(), '--', label='FF Big')
corr = np.corrcoef([ls(temp, 0, -1), ls(temp_ff, 0, -1)])[0, 1]
plt.legend(loc='upper right')
plt.title(f"Cumulative Returns: Return Corr = {corr:.2%}")
plt.grid()
plt.xticks(temp.index[::50], rotation=45)
plt.show()

# EPS

In [ ]:
eps = comp_funq["date cusip epspxq consol".split()]
eps['eps'] = eps.eval("epspxq")
eps = get_non_duplicated_df(eps, "eps")
eps = date_cusip_pivot(eps, "eps").reindex(columns=dfs['ret'].columns)
eps = get_time_merge(eps, dfs['ret'])

In [ ]:
eps_v1 = eps.copy(deep=True)
eps_v1[~idx_Q] = np.nan
eps_portfolio1 = get_sorted_portfolio(
    dfs['ret'], get_sorted_index(
        eps_v1 * get_volume_threshold(date_cusip_pivot(crsp, "vol"), 0.2),
        holding_period=3
        #,quantiles=np.array([0.2, 0.4, 0.6, 0.8])
        #,quantiles=np.array([.3, .7])
        )
)
plot_group_cum(eps_portfolio1, "Absolute EPS")
plot_solo_cum(ls(eps_portfolio1, -1, 0))

In [ ]:
eps_v2 = eps.copy(deep=True)
eps_v2[~idx_Q] = np.nan
eps_v2 = eps_v2.diff(3) / eps_v2.rolling(window=24, min_periods=4).std()
eps_portfolio2 = get_sorted_portfolio(
    dfs['ret'], get_sorted_index(
        eps_v2 * get_volume_threshold(date_cusip_pivot(crsp, "vol"), 0.20),
        holding_period=3
        #,quantiles=np.array([0.2, 0.4, 0.6, 0.8])
        #,quantiles=np.array([.3, .7])
        )
)
plot_group_cum(eps_portfolio2, "EPS Quarter Change")
plot_solo_cum(ls(eps_portfolio2, -1, 0))

# PM

In [ ]:
pm = comp_funq["date cusip niq saleq consol".split()]
pm['pm'] = pm.eval("niq/saleq")
pm = get_non_duplicated_df(pm, "pm")
pm = date_cusip_pivot(pm, "pm").reindex(columns=dfs['ret'].columns)
pm = get_time_merge(pm, dfs['ret']).ffill(limit=2)

In [ ]:
pm_v1 = pm.copy(deep=True)
pm_v1[~idx_Q] = np.nan
pm_portfolio = get_sorted_portfolio(
    dfs['ret'], get_sorted_index(
        pm_v1 * get_volume_threshold(date_cusip_pivot(crsp, "vol"), 0.2),
        holding_period=3
        #,quantiles=np.array([0.2, 0.4, 0.6, 0.8])
        #,quantiles=np.array([.3, .7])
        )
)
plot_group_cum(pm_portfolio, "Absolute Profit Margin")
plot_solo_cum(ls(pm_portfolio, -1, 0))

In [ ]:
pm_v2 = pm.copy(deep=True)
pm_v2[~idx_Q] = np.nan
pm_v2 = pm_v2.diff(3) / pm_v2.rolling(window=18, min_periods=4).std()
pm_v2[count_non_cols(pm_v2) < 100] = np.nan
pm_portfolio2 = get_sorted_portfolio(
    dfs['ret'], get_sorted_index(
        pm_v2 * get_volume_threshold(date_cusip_pivot(crsp, "vol"), 0.2),
        holding_period=3
        #,quantiles=np.array([0.2, 0.4, 0.6, 0.8])
        #,quantiles=np.array([.3, .7])
        )
)
plot_group_cum(pm_portfolio2, "Profit Margin Quarter Change")
plot_solo_cum(ls(pm_portfolio2, -1, 0), "Profit Margin Quarter Change LS (High-Low) Cumulative Return")

In [ ]:
pm_v3 = pm.copy(deep=True)
pm_v3[~idx_Q] = np.nan
pm_v3 = pm_v3.diff(12) / pm_v3.rolling(window=24, min_periods=4).std()
pm_portfolio3 = get_sorted_portfolio(
    dfs['ret'], get_sorted_index(
        pm_v3 * get_volume_threshold(date_cusip_pivot(crsp, "vol"), 0.2),
        holding_period=3
        #,quantiles=np.array([0.2, 0.4, 0.6, 0.8])
        #,quantiles=np.array([.3, .7])
        )
)
plot_group_cum(pm_portfolio3, "Profit Margin Annual Change")
plot_solo_cum(ls(pm_portfolio3, -1, 0))